# Benchmark CLIP vs SigLIP trên MS COCO (Karpathy test)

Notebook này chạy **toàn bộ pipeline** trên Kaggle GPU:
1. Clone repo (có thể đổi `BRANCH` nếu cần test trước khi merge).
2. Tải annotations Karpathy (`dataset_coco.json`) — tự verify đúng 5.000 ảnh test / 25.000 captions.
3. Tự dò thư mục ảnh `val2014` trong `/kaggle/input` rồi symlink vào repo.
4. Verify đủ 5000 ảnh trước khi chạy.
5. Chạy benchmark CLIP và SigLIP, lưu kết quả JSON.

**Chuẩn bị:** Add Data → thêm dataset `nadaibrahim/coco2014` (hoặc `awsaf49/coco-2014-dataset-images`),
và bật GPU: *Settings → Accelerator → GPU T4 x2 (hoặc P100)*. Sau đó **Run All**.

Kết quả cuối: `results/benchmark.json` (nằm trong `/kaggle/working` nên được lưu vào output của notebook).

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nghon4maeri/clip-siglip-semantic-retrieval.git"
BRANCH = "main"  # đổi thành tên branch nếu cần test trước khi merge
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "clip-siglip-semantic-retrieval"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("Repo  :", REPO_DIR)
print("Commit:", commit)

In [ ]:
import transformers

major, minor = map(int, transformers.__version__.split(".")[:2])
if (major, minor) < (4, 36):
    print(f"transformers {transformers.__version__} quá cũ, đang nâng cấp...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.36"], check=True)
else:
    print(f"transformers {transformers.__version__} — OK (cần >= 4.36 cho SigLIP)")

In [ ]:
!python datasets/download_annotations.py

In [ ]:
import glob

# Tự dò thư mục chứa ảnh COCO val2014 trong /kaggle/input (path khác nhau tùy dataset).
candidates = set()
for pattern in ("/kaggle/input/*/val2014", "/kaggle/input/*/*/val2014", "/kaggle/input/*/*/*/val2014"):
    candidates.update(glob.glob(pattern))

def has_coco_images(path):
    try:
        return any(f.startswith("COCO_val2014") and f.endswith(".jpg") for f in os.listdir(path)[:100])
    except OSError:
        return False

candidates = sorted(p for p in candidates if has_coco_images(p))
if not candidates:
    raise FileNotFoundError(

        "Không tìm thấy thư mục val2014 trong /kaggle/input. "
        "Hãy Add Data dataset COCO 2014 (ví dụ: nadaibrahim/coco2014) rồi chạy lại cell này."
    )

image_source = Path(candidates[0])
target = REPO_DIR / "datasets" / "coco" / "images" / "val2014"
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
target.parent.mkdir(parents=True, exist_ok=True)
target.symlink_to(image_source)

n_files = sum(1 for f in os.listdir(image_source) if f.endswith(".jpg"))
print(f"Ảnh nguồn : {image_source} ({n_files} file .jpg)")
print(f"Symlink   : {target} -> {image_source}")

In [ ]:
!python scripts/verify_data.py

In [ ]:
# Chạy benchmark: mặc định cả 2 model với checkpoint B/16, protocol 5 captions/ảnh.
# Thêm --limit N nếu chỉ muốn chạy thử.
!python src/evaluation/benchmark.py --models clip,siglip --batch-size 64 --output results/benchmark.json

In [ ]:
import json

with open("results/benchmark.json", encoding="utf-8") as f:
    res = json.load(f)

print(f"git commit       : {res['git_commit']}")
print(f"annotations sha  : {res['annotation_sha256']}")
print(f"device           : {res['device']}")
print(f"dataset          : {res['dataset']}")
print()
header = f"{'Model':<8} {'i2t R@1':>8} {'i2t R@5':>8} {'i2t R@10':>9} {'t2i R@1':>8} {'t2i R@5':>8} {'t2i R@10':>9} {'Time(s)':>8} {'VRAM(MB)':>9}"
print(header)
print("-" * len(header))
for name, r in res["results"].items():
    m = r["metrics"]
    print(f"{name:<8} {m['i2t_R@1']:>8.2f} {m['i2t_R@5']:>8.2f} {m['i2t_R@10']:>9.2f} "
          f"{m['t2i_R@1']:>8.2f} {m['t2i_R@5']:>8.2f} {m['t2i_R@10']:>9.2f} "
          f"{r['timing']['total_seconds']:>8.1f} {str(r['peak_vram_mb']):>9}")

### Ghi chú

- Đây là **inference/benchmark** trên model pretrained, **không train**. Kaggle chỉ cung cấp GPU để chạy đánh giá.
- Số liệu chỉ có giá trị khi chạy đủ 5000 ảnh (không dùng `--limit`) và với đúng file annotations đã verify.
- Đổi checkpoint: thêm `--clip-checkpoint <hf-id>` / `--siglip-checkpoint <hf-id>`.
- File kết quả `results/benchmark.json` được lưu cùng output của notebook.